[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/Online_Learning_and_Regret.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Online Learning & Regret

The theory that unifies this curriculum's two halves: [adaptive filtering](./Intro_AdFilt_APA.ipynb) *is* online learning, and **regret** — how much worse you did than the best fixed strategy chosen in hindsight — is the guarantee those workshops never stated. Three sessions, ending with LMS getting the theorem it always deserved. Guarantees hold even when the data is *adversarial*: no statistics required.

## 1. Pre-requisites

- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) (convexity, gradients).
- [APA workshop](./Intro_AdFilt_APA.ipynb) — the algorithms about to receive theory.
- [Reinforcement Learning](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) S1 — bandits are online learning with *partial* feedback.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Prediction with Experts & Hedge* (~40 min)
**Goal:** beat the best expert in hindsight — even against an adversary — via multiplicative weights.
**Feeds into:** Session 2 (online gradient descent).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Prediction with Experts & Hedge</b></summary>

**Timing (~40 min).** 10 min the setup and what "adversarial" really means · 8 min why regret is the right thing to measure · 10 min multiplicative weights · 8 min the demo, honestly framed · 4 min buffer.

**Open with the shock.** State the setting fully: the loss sequence may be chosen by an adversary who has read your source code and knows your random seed. Ask the room whether *any* guarantee is possible. The instinct is no — and the instinct is right about absolute performance. The move that rescues everything is changing the question from "will I do well?" to "will I do nearly as well as the best fixed strategy in hindsight?" Getting students to feel that reframing is the session.

**Regret is a relative promise — insist on it.** Regret can be small because you did well, or because *everything* did badly and you matched the field. Hedge never promises good absolute loss; it promises you will not be badly beaten by any single expert you could have committed to. Students consistently over-read the guarantee, and the demo below is a case where regret is tiny for exactly the second reason.

**Board first.** Write $w_i \mathrel{*}= e^{-\eta \ell_i}$ and unpack the two knobs. Exponential, so a bad round costs an expert multiplicatively rather than additively — nobody is eliminated, but reputations collapse fast. And $\eta$ trades off: large $\eta$ reacts quickly and overfits noise, small $\eta$ is stable and slow. Then show $\eta = \sqrt{8\ln N/T}$ in the code and note that the optimal setting *depends on the horizon* — a real limitation, fixed in practice by the doubling trick or anytime variants.

**The $\ln N$ is the punchline.** Regret grows with the *logarithm* of the number of experts, so going from 16 experts to 16,000 costs a factor of about 2.4. Ask what that buys you: you can throw in every strategy you can think of, at almost no cost, and still nearly match whichever turns out to be best. That is why multiplicative weights shows up in boosting, in game theory, and in linear programming — the same algorithm, three literatures.

**Frame the demo honestly — this matters.** The loss matrix here is symmetric: each of the 16 experts is perfect for exactly one 125-round season and useless otherwise, so *every* expert ends with cumulative loss 1875. Hedge's regret of 1.6 against a bound of 52.7 therefore does **not** show Hedge being 33× better than the theory. It shows there was no good expert to find. Even fixed uniform weights would have tied the best expert here. Say this out loud; the debrief works through it.

**Better live variant if you have five minutes.** Make one expert genuinely good — say `losses[:, 3] = 0.2` throughout — and re-run. Now Hedge's weight concentrates on expert 3 and the regret curve means something. Contrasting the two runs teaches more about what regret measures than either alone.

**Ask the room.** "Why exponential weights rather than just picking the expert with the lowest loss so far?" Because *follow-the-leader* is exactly what an adversary punishes: it can make whoever currently leads lose next round, forever. Hedge's randomised, smoothly-hedged weighting is what denies the adversary that lever.
</details>

## 2. The Experts Problem

💡 **Intuition.** Each day, $N$ 'experts' make predictions; you must combine them; then losses are revealed — possibly chosen by an **adversary who read your algorithm**. You cannot always be right, but you can guarantee to nearly match the best single expert *in hindsight*. **Hedge** does it with multiplicative weights: $w_i \mathrel{*}= e^{-\eta \ell_i}$ — exponentially discredit whoever erred. The guarantee $\text{Regret} \le \sqrt{T \ln N / 2} \cdot 2$-ish is *distribution-free*: no probability assumptions anywhere, a totally different kind of promise than [estimation theory's](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [2]:
# Hedge vs an ADVERSARIAL loss sequence (designed to punish any single expert)
T_steps, N_exp = 2000, 16
eta = np.sqrt(8 * np.log(N_exp) / T_steps)

# adversary: each expert is good in its own recurring 'season', terrible otherwise
losses = np.ones((T_steps, N_exp))
for t in range(T_steps):
    losses[t, (t // 125) % N_exp] = 0.0

w = np.ones(N_exp) / N_exp
alg_loss, w_hist = [], []
for t in range(T_steps):
    alg_loss.append(w @ losses[t])
    w = w * np.exp(-eta * losses[t]); w /= w.sum()
    if t % 100 == 0: w_hist.append(w.copy())

cum_alg = np.cumsum(alg_loss)
cum_best = np.min(np.cumsum(losses, 0), 1)
regret = cum_alg - cum_best
bound = np.sqrt(np.arange(1, T_steps+1) * np.log(N_exp) / 2)

plt.figure(figsize=(8, 2.8))
plt.plot(regret, label="Hedge's actual regret")
plt.plot(bound, "k--", linewidth=1, label="√(T ln N / 2) bound")
plt.legend(); plt.xlabel("round"); plt.ylabel("regret vs best expert in hindsight")
plt.title("adversarial losses, yet regret grows only like √T — the average extra loss → 0")
plt.tight_layout(); plt.show()
print(f"final regret {regret[-1]:.1f} ≤ bound {bound[-1]:.1f}; per-round excess {regret[-1]/T_steps:.4f} → 0")
assert regret[-1] <= bound[-1] + 1e-9

final regret 1.6 ≤ bound 52.7; per-round excess 0.0008 → 0


/tmp/ipykernel_2971749/1530568414.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Final regret **1.6** against a proven bound of **52.7**, with per-round excess loss of 0.0008 and falling. The `assert` passed, so the theorem held — and the curve stays well under the dashed $\sqrt{T \ln N / 2}$ line throughout.

**But read this result carefully, because the headline number flatters us.** Look at how the loss matrix is built: `losses[t, (t // 125) % N_exp] = 0.0` over 2000 rounds with 125-round seasons gives exactly 16 seasons for 16 experts, so *every* expert is perfect for one season and useless for the other fifteen. By symmetry all sixteen finish with identical cumulative loss, $2000 - 125 = 1875$. The "best expert in hindsight" is therefore no better than the worst, and even a fixed uniform weighting would have paid $2000 \times (1 - 1/16) = 1875$ — tying the best expert exactly, with no algorithm at all.

So regret of 1.6 does not show Hedge outperforming its bound by 33×. It shows this particular sequence has nothing to learn: there is no good expert to concentrate on, and matching the field is trivial. The bound is loose here because it is a *worst-case* guarantee, and this instance is far from the worst case — not because the analysis is weak.

**What the demo does legitimately establish** is the shape of the promise. Regret grows like $\sqrt{T}$ while the horizon grows like $T$, so the *average* excess loss goes to zero: whatever the sequence, per-round performance converges to the best fixed expert's. That is the meaningful statement, and the $0.0008 \to 0$ figure is the one to point at.

To see Hedge actually working, break the symmetry — give one expert a low loss every round (`losses[:, 3] = 0.2`) and re-run. The weights then concentrate on that expert within a few hundred rounds and the regret curve reflects a genuine identification problem. Comparing the two runs is the clearest way to see that regret measures *relative* performance, and that a small number can mean "we did well" or merely "there was nothing to be gained."

---
### 🕐 Session 2 of 3 — *Online Gradient Descent & the Regret Bound* (~40 min)
**Goal:** prove the O(√T) regret of OGD — the cleanest nontrivial proof in machine learning.
**Builds on:** Session 1; [Optimization](../Intro_Math/Optimization/Optimization.ipynb). &nbsp; **Feeds into:** Session 3 (the adaptive-filtering reunion).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Online Gradient Descent & the Regret Bound</b></summary>

**Timing (~40 min).** 5 min the setting · 20 min the proof, slowly · 10 min the demo and negative regret · 5 min buffer. The proof is the session — it is short enough to do completely, and doing it completely is the point.

**This is the best proof in the workshop; teach it as one.** Three lines, no measure theory, no statistics, and it yields a real guarantee against an adversary. Tell the room in advance that they will see the entire argument with nothing waved away — that is rare and it earns their attention.

**Walk the potential argument in four beats.** (1) Track $\|w_t - u\|^2$, the distance to an arbitrary comparator — this is the potential, and choosing it is the only creative step. (2) Expand one update; the cross term $-2\eta\nabla_t^\top(w_t - u)$ appears. (3) Convexity converts that cross term into exactly the per-round regret, $\ell_t(w_t) - \ell_t(u) \le \nabla_t^\top(w_t - u)$ — this is where convexity is spent, and it is spent once. (4) Sum, and the potential telescopes to $\|w_1 - u\|^2$. Ask which step used which hypothesis; every assumption is load-bearing and visibly so.

**Where the $\sqrt{T}$ comes from.** Do not let this be algebra. The bound is $\frac{R^2}{2\eta} + \frac{\eta G^2 T}{2}$: the first term is the price of starting in the wrong place and shrinks with larger steps, the second is the price of taking steps at all and grows with them. Balancing the two forces $\eta \sim 1/\sqrt{T}$ and gives $RG\sqrt{T}$. The $\sqrt{T}$ is the fingerprint of a trade-off, and students who see that can re-derive the rate rather than recall it.

**The projection detail.** One line of the proof says "projection only shrinks distance," which is true because we project onto a *convex* set and the comparator $u$ is in it. Worth ten seconds — it is the only place the radius-$R$ ball matters, and it is why the constraint set has to be convex.

**Misconception — the important one.** "Regret going negative means the bound was violated." It does not. The bound is one-sided: regret is guaranteed to stay *below* $RG\sqrt{T}$, with no lower limit. Negative regret means OGD beat every fixed comparator, which is possible whenever the environment is non-stationary. Get this said before you run the cell, since the printed result is $-1251$ and someone will read it as a failure.

**Ask the room.** "The targets switch every 1000 rounds. What is the best *fixed* $u$ over the whole stream?" Something mediocre averaging three unrelated regimes — good for none of them. OGD is not bound by that: it re-converges after each switch. This is also the honest limitation of the framework, and worth naming: comparing against the best fixed strategy is a weak benchmark in a changing world, which is exactly why *dynamic regret*, comparing against the best changing sequence, exists.

**If you have time.** Ask what happens with $\eta$ far too large or far too small; the two terms of the bound predict both failures — oscillation versus never arriving — before you run anything.
</details>

## 3. OGD and Its Guarantee

Setting: at each round pick $w_t$, adversary reveals convex loss $\ell_t$, you pay $\ell_t(w_t)$ and update $w_{t+1} = w_t - \eta \nabla \ell_t(w_t)$ (projected into a radius-$R$ ball).

**Theorem.** For convex losses with $\|\nabla \ell_t\| \le G$: with $\eta = \frac{R}{G\sqrt{T}}$,
$$\text{Regret}_T = \sum_t \ell_t(w_t) - \min_{\|u\|\le R} \sum_t \ell_t(u) \;\le\; RG\sqrt{T}.$$

**Proof** (three lines — the famous potential argument). Let $u$ be any comparator; expand the 'distance potential':
$$\|w_{t+1} - u\|^2 \le \|w_t - \eta \nabla_t - u\|^2 = \|w_t - u\|^2 - 2\eta \nabla_t^T(w_t - u) + \eta^2\|\nabla_t\|^2$$
(projection only shrinks distance). Convexity gives $\ell_t(w_t) - \ell_t(u) \le \nabla_t^T (w_t - u)$; substitute and sum over $t$ — the potential terms *telescope*:
$$\text{Regret}_T \le \frac{\|w_1 - u\|^2}{2\eta} + \frac{\eta}{2}\sum_t \|\nabla_t\|^2 \le \frac{R^2}{2\eta} + \frac{\eta G^2 T}{2}.$$
Optimize $\eta$ → $RG\sqrt{T}$. $\blacksquare$ No statistics, no stationarity — the data may be chosen by a demon and the bound still holds.

In [3]:
# Watch the theorem hold on adversarial online linear regression
d, T_steps = 5, 3000
R, G = 2.0, 6.0
u_seasonal = [rng.standard_normal(d) for _ in range(3)]     # the world switches targets!

w = np.zeros(d)
eta = R / (G * np.sqrt(T_steps))
reg_terms, losses_alg, all_grads = [], [], []
X_hist, y_hist = [], []
for t in range(T_steps):
    x = rng.standard_normal(d); x /= max(1, np.linalg.norm(x)/1.5)
    u_now = u_seasonal[(t // 1000) % 3]
    y = u_now @ x + 0.1*rng.standard_normal()
    pred = w @ x
    losses_alg.append((pred - y)**2 / 2)
    g = (pred - y) * x
    w = w - eta * g
    if np.linalg.norm(w) > R: w *= R / np.linalg.norm(w)    # projection
    X_hist.append(x); y_hist.append(y)

# best FIXED comparator in hindsight (least squares over the whole stream, norm-capped)
Xh, yh = np.array(X_hist), np.array(y_hist)
u_star, *_ = np.linalg.lstsq(Xh, yh, rcond=None)
if np.linalg.norm(u_star) > R: u_star *= R / np.linalg.norm(u_star)
loss_best = ((Xh @ u_star - yh)**2 / 2)

regret = np.cumsum(losses_alg) - np.cumsum(loss_best)
plt.figure(figsize=(8, 2.8))
plt.plot(regret, label="OGD regret vs best fixed w (hindsight)")
plt.plot(R*G*np.sqrt(np.arange(1, T_steps+1)), "k--", linewidth=1, label="RG√T")
plt.legend(); plt.xlabel("round")
plt.title("the world switched targets twice: regret goes NEGATIVE — OGD adapts while any fixed comparator must average over regimes")
print(f"final regret {regret[-1]:.0f} ≤ RG√T = {R*G*np.sqrt(T_steps):.0f}")
print("negative regret is not a bug: the bound only PROMISES ≤ RG√T; against a shifting world,")
print("adapting beats any fixed strategy, so the realized regret can dip below zero.")

final regret -1251 ≤ RG√T = 657
negative regret is not a bug: the bound only PROMISES ≤ RG√T; against a shifting world,
adapting beats any fixed strategy, so the realized regret can dip below zero.


**What just happened.** Final regret **−1251** against a bound of $RG\sqrt{T} = 657$. The theorem is satisfied, comfortably — and the reason the numbers look strange is worth being precise about.

**The bound is one-sided.** It promises regret $\le RG\sqrt{T}$. It says nothing whatsoever about a floor. Negative regret is not a violation, a bug, or a fluke: it means OGD accumulated *less* total loss than the best single fixed $w$ chosen with full hindsight. Students routinely misread the inequality as a two-sided prediction, so it is worth writing $-1251 \le 657$ on the board and letting the arithmetic settle it.

**Why it happened here.** The world switches targets twice, cycling through three unrelated $u$ vectors. Any *fixed* comparator must serve all three regimes with one weight vector, so `lstsq` returns a compromise that is mediocre everywhere — good for none of the three. OGD is under no such constraint: after each switch it simply re-converges over the following few hundred rounds and spends most of its time well fitted to whatever regime is current. Adaptivity beats commitment when the environment moves, and the negative regret is that advantage measured.

**The honest reading of this is a limitation, not a victory.** Regret against the best fixed strategy is a *weak benchmark* in a non-stationary world — so weak that a mediocre adaptive algorithm can beat it. The right response is not to celebrate the negative number but to notice the yardstick has stopped being informative. The literature's answer is **dynamic regret**, which compares against the best *sequence* $u_1, \dots, u_T$ rather than a single fixed $u$, with bounds that degrade gracefully as a function of how much the comparator moves. That is the framework you actually want for tracking, and it is what Session 3's adaptive-filtering reunion is implicitly reaching for.

**What remains genuinely impressive.** Nothing in the proof assumed the data was stationary, Gaussian, independent, or generated by any process at all — the targets could have been chosen by an adversary reading this code, and the $RG\sqrt{T}$ ceiling would still hold. That is a categorically different kind of promise from the statistical guarantees in [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb), which are sharper but evaporate the moment their assumptions fail. Here the assumptions are convexity and a bounded gradient, and that is all.

---
### 🕐 Session 3 of 3 — *The Adaptive-Filtering Reunion* (~35 min)
**Goal:** LMS = OGD on squared loss: restate the filtering workshops as regret guarantees.
**Builds on:** Session 2; [APA](./Intro_AdFilt_APA.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Adaptive-Filtering Reunion</b></summary>

**Timing (~35 min).** 10 min the symbol-for-symbol identification · 8 min what the regret guarantee adds to the classical story · 10 min the demo with its mid-stream switch · 7 min the wider map and buffer.

**Do the derivation live — do not just assert it.** Put $\ell_t(w) = \tfrac12(d_t - w^\top x_t)^2$ on the board and differentiate: $\nabla \ell_t = -(d_t - w^\top x_t)\,x_t = -e_t x_t$. Substitute into the OGD update $w \leftarrow w - \eta\nabla$ and you get $w \leftarrow w + \eta\, e_t x_t$. That is LMS, and the room should watch it appear rather than be told it does. This is the moment the workshop exists for.

**Then say what it buys.** The [adaptive filtering workshops](./Intro_AdFilt_APA.ipynb) presented LMS as a heuristic — a stochastic approximation to steepest descent on a cost surface, analysed under assumptions of stationarity, independence, and eigenvalue conditions on $R$. Session 2's theorem hands the *same algorithm* a guarantee that needs none of that. LMS was never merely a heuristic; it just had not been given the right yardstick.

**Frame the two views as complementary, not as a winner.** Statistics gives sharper answers when its assumptions hold — Wiener theory tells you the exact optimal filter and the exact misadjustment, which regret cannot. Regret gives unbreakable answers when the assumptions fail. Students like to rank the two; the useful skill is knowing which is live in a given problem. Ask when you would trust each.

**Misconception.** "The regret bound tells you LMS converges to the true filter." It does not. It bounds *cumulative loss* relative to the best fixed filter in hindsight; it says nothing about parameter identification, and there are streams where regret is small while the weights never settle anywhere near $w_{\text{true}}$. Prediction and identification are different goals, and the guarantee is about the former.

**On the demo's negative result.** The system flips sign at the halfway point, so no fixed filter can serve both halves — the hindsight-optimal $u^\star$ is a compromise that is bad throughout. Average regret comes out at $-5.5$. As in Session 2, this is a comment on the benchmark rather than a triumph, and you should say so. If someone asks what a *fair* comparison looks like, that is dynamic regret, and this demo is the cleanest motivation for it in the workshop.

**Close on the map, briefly.** NLMS is OGD with per-round step normalisation; [RLS](./Intro_RLS.ipynb) is online Newton, which achieves $O(\log T)$ regret on strongly convex losses rather than $O(\sqrt{T})$; tracking drifting systems is dynamic regret. The whole adaptive-filtering ladder is one family of online convex optimisation algorithms, and this is the sentence that unifies the two halves of the curriculum. It is the note to end the workshop on.
</details>

## 4. LMS Gets Its Theorem

💡 **Intuition.** Look again at OGD on the squared loss $\ell_t(w) = \tfrac12(d_t - w^T x_t)^2$: the update is $w \mathrel{+}= \eta \, e_t x_t$ — **that is LMS, symbol for symbol**. So the entire regret machinery transfers: LMS is guaranteed to track within $O(\sqrt{T})$ of the best fixed filter *chosen after seeing all the data*, with no stationarity, no Gaussian noise, no eigenvalue conditions — a guarantee the [Wiener-theory story](../Intro_DSP/Statistical_Signal_Processing.ipynb) cannot make when its assumptions fail. The two views are complementary: statistics gives *sharper* answers when its assumptions hold; regret gives *unbreakable* ones when they don't. (NLMS ≈ per-round step normalization; [RLS](./Intro_RLS.ipynb) ≈ online Newton and can achieve $O(\log T)$ regret on strongly-convex losses; tracking *drifting* systems is 'dynamic regret'.)

In [4]:
# The same system-ID scenario as the APA workshop — but scored by REGRET, and with a
# mid-stream system SWITCH that violates every stationarity assumption.
M, T_steps = 8, 6000
w_true1 = rng.standard_normal(M); w_true2 = -w_true1
x_in = rng.standard_normal(T_steps + M)

w = np.zeros(M); losses_lms = []
X_rows, d_hist = [], []
eta = 0.02
for t in range(T_steps):
    xv = x_in[t:t+M][::-1]
    w_now = w_true1 if t < T_steps//2 else w_true2         # the switch
    d = w_now @ xv + 0.05*rng.standard_normal()
    e = d - w @ xv
    losses_lms.append(0.5 * e**2)
    w = w + eta * e * xv                                    # LMS = OGD, verbatim
    X_rows.append(xv); d_hist.append(d)

Xh, dh = np.array(X_rows), np.array(d_hist)
u_star, *_ = np.linalg.lstsq(Xh, dh, rcond=None)            # best FIXED filter in hindsight
loss_best = 0.5*(Xh @ u_star - dh)**2
regret = np.cumsum(losses_lms) - np.cumsum(loss_best)

plt.figure(figsize=(8, 2.8))
plt.plot(regret / np.arange(1, T_steps+1), label="LMS per-round regret vs best fixed filter")
plt.axhline(0, color="k", linewidth=0.8)
plt.legend(); plt.xlabel("round")
plt.title("system flips sign mid-stream: LMS's AVERAGE regret still → 0 (and can go negative —\nno fixed filter handles both halves, but LMS adapts to each)")
plt.tight_layout(); plt.show()
avg = regret[-1]/T_steps
print(f"average regret per round at T: {avg:.3f} — strongly NEGATIVE: no fixed filter can serve")
print("both halves of the stream, so adaptive LMS beats the best of them outright.")
print("The guarantee (≤ O(1/√T) above zero) is intact; beating it is the bonus adaptivity buys.")

average regret per round at T: -5.547 — strongly NEGATIVE: no fixed filter can serve
both halves of the stream, so adaptive LMS beats the best of them outright.
The guarantee (≤ O(1/√T) above zero) is intact; beating it is the bonus adaptivity buys.


/tmp/ipykernel_2971749/706582770.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The line `w = w + eta * e * xv` is LMS exactly as the [adaptive filtering workshops](./Intro_AdFilt_APA.ipynb) wrote it — and it is also online gradient descent on the squared loss, symbol for symbol, since $\nabla_w \tfrac12(d - w^\top x)^2 = -e\,x$. Nothing was reimplemented for this session. The same code now simply gets scored by a different yardstick.

That reframing is the point of the workshop. LMS was introduced as a heuristic: a stochastic approximation to gradient descent on a cost surface, whose analysis required stationarity, independence assumptions, and eigenvalue conditions on the input autocorrelation. Session 2's three-line proof hands that identical algorithm a guarantee requiring none of it — only convexity and a bounded gradient. LMS always had a theorem; it was being analysed with the wrong tools.

**The switch is what makes the demo bite.** Halfway through, the system flips to $-w_{\text{true}}$, violating stationarity as completely as it can be violated. Classical Wiener analysis has nothing to say about this stream — there is no single optimal filter to converge to. The regret guarantee is untouched, because it never assumed one existed.

**Average regret came out at −5.5, and again the negative sign is about the benchmark.** No fixed filter can serve a stream that inverts halfway: the hindsight-optimal $u^\star$ from `lstsq` averages the two regimes into something close to useless for both. LMS tracks each half and beats that compromise outright. So the number confirms adaptivity is valuable here — and simultaneously shows that "best fixed filter" has stopped being a meaningful comparison. The guarantee (average regret $\le O(1/\sqrt{T})$ above zero) holds and is not the interesting part; the interesting part is that the yardstick has gone slack, which is precisely the motivation for **dynamic regret**, where the comparator is allowed to move.

**One caution on what is being promised.** The bound concerns cumulative *prediction loss* relative to a fixed comparator. It does not claim the weights converge to $w_{\text{true}}$, and after the switch they demonstrably chase a moving target. Good prediction and correct identification are different goals, and only the first is covered here.

Taken together with Sessions 1 and 2: Hedge, OGD, LMS, NLMS, and [RLS](./Intro_RLS.ipynb) are one family of online convex optimisation algorithms differing in how the update is preconditioned — RLS being the online-Newton end, which buys $O(\log T)$ regret on strongly convex losses. The signal processing half and the machine learning half of this curriculum have been studying the same subject in different notation.

## 5. Conclusion

Hedge beats hindsight's best expert against adversaries; OGD's three-line telescope gives $RG\sqrt{T}$; and LMS turns out to have been online gradient descent all along — carrying a distribution-free guarantee the classical story never mentioned. The curriculum's two halves were one subject.

---
## Where next

- [Concentration Inequalities](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) — the statistical guarantees, for when assumptions *do* hold.
- [Reinforcement Learning](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) — bandits: regret with partial feedback.
- [RLS](./Intro_RLS.ipynb) — the online-Newton end of the spectrum.